# 03_review — formal governance review template

Select profiled data, author governance enrichment, author guardrail rules, then complete formal governance review.

02_pipeline may create draft, pending, or active-pending-review records during engineering work. Formal approval, rejection, replacement, and deactivation happen only in 03_review.


## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release  | Tested by | Date tested | 
|---|---|---| 
| v0.1.0 |  Voyce| 13 Jul 2026 | 


## 1. Run `00_env_config`


In [ ]:
%run 00_env_config


## 2. Import governance widgets


In [ ]:
from fabricops_kit import (
    widget_render_data_agreement,
    widget_render_data_steward,
    widget_view_agreement_catalogue,
    # FabricOps v0.1.0 onwards
    widget_select_guardrail_target,
    widget_enrich_table_metadata,
    widget_author_schema_freshness_profile_rules,
    widget_author_dq_rules,
    widget_review_guardrail_governance,
)


## 3. Select or create the Data Steward

Run this standalone cell first to resolve the steward records used by the Data Agreement workflow.


In [ ]:
steward_widget = widget_render_data_steward(spark=spark)


## 4. Select or create the Data Agreement

Resolve the Data Agreement after its provider and recipient Data Stewards are available.


In [ ]:
agreement_widget = widget_render_data_agreement(spark=spark)


## 5. Review the selected agreement's Data Contract

The selected agreement is resolved through `METADATA_DATA_CONTRACT`, so only linked logical datasets are visible. Contract membership is shared across Development and Production, while catalogue and evidence observations remain separate and the active environment determines which observations are reviewed.

To review Development or Production evidence, run this notebook under that active environment's `00_env_config`; this viewer does not browse both environments together. The Data Steward selection belongs to agreement selection and is not a separate viewer filter.


In [ ]:
governance_catalogue_view = widget_view_agreement_catalogue(
    agreement=agreement_widget,
    target="metadata",
    schema=METADATA_SCHEMA,
    spark_session=spark,
)


Load the selected catalogue and profile DataFrames in the following native Fabric result cells.


In [ ]:
views = governance_catalogue_view["get_views"]()
catalogue_df = views["catalogue"]
profile_df = views["profile"]
frequency_df = views["frequency"]


In [ ]:
display(catalogue_df)


In [ ]:
display(profile_df)
display(frequency_df)


## 6. Select profiled data

Select a profiled source table or pipeline output from the metadata catalogue. Run 02_pipeline profiling first if the table or output is not available for selection.


In [ ]:
guardrail_target_state = widget_select_guardrail_target(
    spark_session=spark,
)


## 7. Author enrichment

Review and update column context, classification, and other enrichment details for the selected table.

Engineering users can draft enrichment in 02_pipeline. Governance users can author or refine enrichment here before formal review.


In [ ]:
enrichment_state = widget_enrich_table_metadata(
    spark_session=spark,
)


## 8. Author guardrail rules

Create or update schema, freshness, profile-behaviour, and data-quality guardrail rules for the selected table.

Rules authored here are governance-owned. Rules drafted in 02_pipeline can be reviewed and finalised here.


In [ ]:
schema_freshness_profile_state = widget_author_schema_freshness_profile_rules(
    guardrail_target_state,
    spark_session=spark,
    source_notebook_type="03_review",
    created_by_role="governance",
)

dq_authoring_state = widget_author_dq_rules(
    guardrail_target_state,
    spark_session=spark,
    source_notebook_type="03_review",
    created_by_role="governance",
)


## 9. Review governance records

Review enrichment and guardrail records for the selected table.

Approve, reject, replace, deactivate, or inspect history for records that require a formal governance decision.


In [ ]:
governance_review_state = widget_review_guardrail_governance(
    guardrail_target_state,
    spark_session=spark,
)
